# Chatroom App

Here we create a multi-user **Chatroom App** using Flet. Chat message are broadcasted using the built-in PubSub library and we will show how to enhance the UI with reusable controls and animations. We will store state (chat history & user info), in-memory for simplicity. This should be a good starting point for creating more complex projects.

## Broadcasting with PubSub

<video
  src="./img/flet-chat/v1.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

This run using `flet run --web v1.py` and we open the same link in separate browsers creating two sessions.

```python
import flet as ft
from dataclasses import dataclass

@dataclass
class Message:
    user: str
    text: str

@ft.component
def AppView():
    page = ft.context.page
    session_id = page.session.id
    history, set_history = ft.use_state([])     # <1>
    message, set_message = ft.use_state("")
    
    def on_message(msg_obj: Message):           # <3>
        page.run_thread(lambda: set_history(lambda h: [*h, msg_obj]))
        page.update()

    # subscribe once. use_effect expects cleanup function
    def subscribe():                                        
        page.pubsub.subscribe(on_message)
        def cleanup(): 
            page.pubsub.unsubscribe(on_message)
        return cleanup

    # empty deps => run once on mount, and cleanup on unmount
    ft.use_effect(subscribe, [])        # <2>

    def send_click(e):  # <4>
        page.pubsub.send_all(msg_obj=Message(user=session_id, text=message))
        set_message("")

    return ft.Column(
        controls=[
            ft.Column(controls=[ft.Text(f"{m.user}: {m.text}") for m in history]),
            ft.Row(controls=[
                ft.TextField(
                    label="New message",
                    value=message,
                    width=400,
                    on_change=lambda e: set_message(e.control.value),
                    on_submit=send_click
                ),
                ft.Button("Send", on_click=send_click)
            ]),
        ]
    )


if __name__ == "__main__":
    ft.run(lambda page: page.render(AppView))
```

1. The history and current message are initialized using `use_state` hook. So that these are persisted across re-renders. 

2. Then, the "effect" `subscribe` is called only once during initialization since the dependencies are empty `[]`. See the docs on [`use_effect`](https://docs.flet.dev/types/useeffect/?h=use_effect). This also returns a cleanup function which unsubscribes from pubsub. The subscribe mechanism makes a FastAPI worker process trigger the function `on_message`  whenever a message is published. Moreover, this works due to having closure on `page`.

3. This calls `set_history` with an update function (note that we can either put a value or an update function with the state variable as input). Here we opted for an update function to ensure that the latest message is captured. Finally, we force update. This is run in the current page's thread (sync)[^run_thread].

4. We've set up the plumbing for getting messages. How about sending? For this we simply use `page.pubsub.send_all`. This takes in any Python object and becomes the input of the functions that was specified during PubSub subscribe. This explains the input type `msg_obj: Message` for `on_message`. We likewise send an object of type `Message`. Finally, the `message` variable is cleared.

[^run_thread]: The exact mechanism is unclear but this is what finally worked to properly run everything in the current render context without UX issues, and no error logs.

## Adding usernames

Here we add usernames which users choose when they join the chatroom. This naturally has to be unique. Hence, we have an `active_users` set of strings. Note that since we have multi-users and the app works in a distributed manner, this adds a layer of complexity in ensuring consistency. In particular, we're wary of **race conditions.** For example, we want to validate that a username is unique, however by the time we have completed creating a user another may have registered the same username, resulting in duplicate usernames.

To solve this, we implement **thread locks** ensuring that only one thread accesses `active_users` at a time.

```python
@dataclass
class ChatRoom:
    active_users: set[str] = field(default_factory=set)
    _lock: threading.Lock = field(default_factory=threading.Lock, repr=False)

    def add_user(self, name: str):
        with self._lock:
            self.validate_username(name)
            self.active_users.add(name)

    def remove_user(self, name: str):
        with self._lock:
            self.active_users.discard(name)

    def validate_username(self, name: str):
        if name in self.active_users:
            raise ValueError(f'"{name}" is already taken. Please choose another.')
        if not name.strip():
            raise ValueError("Username cannot be empty.")
```

Next, we have the **join dialog**. To understand this, let's walk backwards:

```python
def JoinDialog(join_click: Callable, chatroom: ChatRoom):
    def wrap_close(handler: Callable):
        def handle(e):
            entered = username.value.strip()
            try:
                chatroom.add_user(entered)

            except ValueError as error: # <3>
                e.page.pop_dialog()
                rejoin_dialog = JoinDialog(join_click, chatroom)
                error_dialog = ft.AlertDialog(
                    modal=True,
                    title=ft.Text("Invalid Username"),
                    content=ft.Text(str(error)),
                    actions=[
                        ft.Button(
                            "OK", 
                            on_click=lambda ev: (                    
                                ev.page.pop_dialog(),            
                                e.page.show_dialog(rejoin_dialog)
                            )
                        )
                    ],
                    actions_alignment=ft.MainAxisAlignment.END,
                )
                e.page.show_dialog(error_dialog)
                return
            
            e.page.pop_dialog()
            handler(e, entered)     # <2>
        return handle

    username = ft.TextField(label="Enter your name")

    return ft.AlertDialog(
        modal=True, # <1>
        title=ft.Text("Welcome!"),
        content=ft.Column([username], tight=True),
        actions=[ft.Button("Join", on_click=wrap_close(join_click))], 
        actions_alignment=ft.MainAxisAlignment.END
    )
```
1. A **modal dialog** temporarily blocks the user from interacting with the app content until it is dismissed.
The important part here is `wrap_close` which wraps the handler `join_click` from the main program to pass the username obtained entered in the dialog.
2. Ultimately, the modal calls `join_click(e, entered)` where `entered` is the validated username entered by the user.
3. In case username already exists, we get a `ValueError`. Then, the join dialog is closed, an error dialog is shown,
and a join dialog is opened. A small technical detail is that the error dialog is non-blocking. Hence, we have to exit the handler
once the error modal is displayed (which flows to a rejoin after clicking "OK") by using `return`. Otherwise, it will go to the next 
scope and pop the error dialog just opened.

<video
  src="./img/flet-chat/v2.mp4"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>